In [ ]:
!pip install sentence-transformers

In [ ]:
import pandas as pd
import re
import math
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:

def cosine_similarity(str1: str, str2: str) -> float:
    v1, v2 = model.encode([str(str1), str(str2)])
    dot = np.dot(v1, v2)
    return float(dot / (np.linalg.norm(v1) * np.linalg.norm(v2)))


In [ ]:
def compare_model_outputs(path_without_cot: str, path_with_cot: str, output_path: str):
    df1 = pd.read_csv(path_without_cot)
    df2 = pd.read_csv(path_with_cot)

    assert len(df1) == len(df2), "CSVs have different number of rows"

    results = []
    for i, (row1, row2) in enumerate(zip(df1.itertuples(), df2.itertuples())):
        score = cosine_similarity(row1.output, row2.output)
        results.append({
            "number": row1.number,
            "prompt_instruction": row1.prompt_instruction,
            "attack_instruction": row1.attack_instruction,
            "output_WithoutCOT": row1.output,
            "output_WithCOT": row2.output,
            "cosineSimilarityScore": round(score, 4),
        })
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/700 rows...")

    pd.DataFrame(results).to_csv(output_path, index=False)
    print(f"Done. Results saved to {output_path}")

In [ ]:
compare_model_outputs(
    path_without_cot="without_cot.csv",
    path_with_cot="with_cot.csv",
    output_path="comparison_results.csv",
)